# Dask
This is a tutorial to use the Dask cluster from Jupyter.

In [1]:
# Init environment before running a demo notebook.
from resources.utils import *
init_demo()
init_dask_cluster_staging(scale=2)
init_dask_cluster_eopf(scale=2)
from resources.utils import *  # reload the global vars again

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/69064ab2809647b69538889399bc54f0/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| msgpack | 1.1.0  | 1.0.7     | None    |
| toolz   | 1.0.0  | 0.12.0    | None    |
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/0e74aa0a1ca543fdb140946380abdb9e/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [2]:
# Other imports
import logging
import sys

In [3]:
# My local "./resources" folder contains a "my_dask_utils.py" module.
# I want to be able to use the same "import my_dask_utils" line on both client and workers.
# For this, I'm updating my PYTHONPATH.
sys.path.append("./resources")
import my_dask_utils

# Show my client IP address using a function from this local module
logging.warning(f" Client IP address: {my_dask_utils.get_ip_address()}")

# I want to be able to do the same from the dask workers. 
for client in dask_client_staging, dask_client_eopf:
    
    # First I need to forward logging from dask workers to the client.
    # NOTE: we need to the the logging in the workers, "print" won't be forwarded.
    client.forward_logging()
    
    # Then I need to upload my local module to the dask workers
    client.upload_file("./resources/my_dask_utils.py")

### Implement the `Futures` tutorial: https://docs.dask.org/en/stable/futures.html
Note: this is how the `rs-server-staging` web service is using Dask.

In [4]:
def inc(x, name):

    # From staging workers
    if name == "staging":
        # Just make sure that rs-server-staging is installed inside the dask workers.
        # NOTE: this import doesn't run the staging web service. 
        # It only imports its modules to be able to call the staging functions.
        # This is actually what the staging web service (that runs on another pod) is doing.
        from rs_server_staging.processors import processors

    # From eopf workers, we can also import the eopf modules
    else:
        from eopf.product.eo_product import EOProduct        
    
    # Note that this is run from a dask worker with a different IP than the client,
    # and that the workers also differ between the staging and eopf workers.
    logging.warning(f" Worker IP address for {name!r}: {my_dask_utils.get_ip_address()}")

    return x + 1

def add(x, y):
    return x + y

# Test this for the staging and eopf client
for client, name in ((dask_client_staging, "staging"), (dask_client_eopf, "eopf")):
    print(f"\nTest {name!r}:")
    
    a = client.submit(inc, 10, name)  # calls inc(10) in background thread or process
    b = client.submit(inc, 20, name)  # calls inc(20) in background thread or process
    print(f"a: {a.result()}")
    print(f"b: {b.result()}")

    c = client.submit(add, a, b)  # calls add on the results of a and b
    print(f"c: {c.result()}")

    futures = client.map(inc, range(5), name=name)
    results = client.gather(futures)  # this can be faster
    print(results)


Test 'staging':


a: 11


b: 21
c: 32
[1, 2, 3, 4, 5]

Test 'eopf':


a: 11
b: 21
c: 32
[1, 2, 3, 4, 5]


### 'pip install' inside Dask workers

In [5]:
# Test if a module is installed inside the dask workers
def test_pip():
    import argh # yes this is a real module, see: https://pypi.org/project/argh/
    logging.warning(f" argh methods/attributes: {dir(argh)}")

# The first time you will test this in workers, it will fail
try:
    client.submit(test_pip).result()
except ModuleNotFoundError:
    print("'argh' is not yet installed in the workers ...")

# You can install it with: https://distributed.dask.org/en/stable/plugins.html#built-in-scheduler-plugins
from dask.distributed import PipInstall
plugin = PipInstall(packages=["argh"])
client.register_plugin(plugin)

# Now it will work.
client.submit(test_pip, pure=False).result() # IMPORTANT: use pure=False to disable cache
print("'argh' is now installed in the workers.")

'argh' is not yet installed in the workers ...


'argh' is now installed in the workers.


### Shutdown the dask clusters

In [6]:
# You can scale the clusters to 0 workers
dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)
dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway_staging)
shutdown_dask_clusters(dask_gateway_eopf)

Shutting down cluster '69064ab2809647b69538889399bc54f0' ...
Shutting down cluster '0e74aa0a1ca543fdb140946380abdb9e' ...
